In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import neurokit2 as nk
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import butter, filtfilt, find_peaks

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr, spearmanr
from scipy.integrate import trapezoid
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier
import scipy.signal as scisig
from scipy.signal import resample

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch.nn.functional as F
import onnx

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    matthews_corrcoef, roc_auc_score
)

import warnings
warnings.filterwarnings(
    "ignore",
    module="neurokit2"
)

np.set_printoptions(precision=3, suppress=True)

In [26]:
DATA_PATH = "../data/WESAD"
LABEL_WINDOW = 2
INPUT_WINDOW = 60
SAMPLE_FS = {'ACC': 32, 'BVP': 64, 'TEMP': 4, 'LABEL': 700, 'EDA': 4}
USE_FREQ_HRV = False

SUBJECT_IDS = (
    [f"S{i}" for i in range(2, 12)] +
    [f"S{i}" for i in range(13, 18)]
)

def load_subject(path):
    with open(path, "rb") as f:
        return pickle.load(f, encoding="latin1")
    
def load_survey(path):
    with open(path,"r") as f:
        return f.readlines()

def downsample_label(x, fs_high, fs_low):
    x = np.asarray(x[max(0, (INPUT_WINDOW-LABEL_WINDOW) * fs_high):])
    T = len(x) / fs_high
    n_out = int(round(T * fs_low))

    # time stamps for input samples
    t = np.arange(len(x)) / fs_high

    # bin edges for output samples (uniform in time)
    edges = np.linspace(0, T, n_out + 1)

    y = np.empty(n_out, dtype=int)
    for i in range(n_out):
        mask = (t >= edges[i]) & (t < edges[i+1])

        if mask.any():
            l = np.bincount(x[mask], minlength=8)[1:5].argmax()
            y[i] = l if l != 3 else 0
        else:
            # rare edge case if rounding creates an empty bin
            y[i] = y[i-1] if i > 0 else x[0]
    return y

def upsample(low_signal, high_signal, fs_high, fs_low):
    upsample_factor = fs_high / fs_low

    # Compute target length
    target_len = len(high_signal)

    # Repeat each low-frequency sample
    indices = np.floor(np.arange(target_len) / upsample_factor).astype(int)
    indices = np.clip(indices, 0, len(low_signal) - 1)

    return low_signal[indices]

def filterSignalFIR(data, cutoff=0.4, numtaps=64):
    f = cutoff / (32 / 2.0)
    FIR_coeff = scisig.firwin(numtaps, f)
    return scisig.lfilter(FIR_coeff, 1, data)

def dom_nonzero_freq(signal, fs_hz):
    """
    Returns most dominant non-zero frequency via fft
    """
    T, K, C = signal.shape
    y = np.fft.rfft(signal, axis = 0)
    yf = np.fft.rfftfreq(T, 1/fs_hz)
    mag = np.abs(y)
    mag[0, :, :] = 0
    idx = np.argmax(mag, axis = 0)
    return yf[idx]

def eda_features(eda, fs, corr_method="pearson"):
    """
    Compute SCR/SCL features from an EDA signal.

    Parameters
    ----------
    eda : array-like
        Raw EDA signal (skin conductance).
    fs : float
        Sampling frequency in Hz.
    corr_method : str
        "pearson" or "spearman" for SCL-time correlation.

    Returns
    -------
    features : dict
        Dictionary of computed features.
    signals : pd.DataFrame
        Processed signal dataframe from neurokit2.
    info : dict
        Event metadata returned by neurokit2.
    """
    eda = np.asarray(eda, dtype=float)

    # 1) Clean + decompose + detect SCR peaks
    # signals columns typically include:
    # EDA_Clean, EDA_Tonic (SCL), EDA_Phasic (SCR), SCR_Onsets, SCR_Peaks, SCR_Recovery, ...
    signals, info = nk.eda_process(eda, sampling_rate=fs)

    scl = signals["EDA_Tonic"].to_numpy()   # tonic = SCL
    scr = signals["EDA_Phasic"].to_numpy()  # phasic = SCR

    # Time vector
    t = np.arange(len(eda)) / fs

    # 2) Correlation between SCL and time
    if corr_method.lower() == "pearson":
        scl_time_corr, scl_time_corr_p = pearsonr(t, scl)
    elif corr_method.lower() == "spearman":
        scl_time_corr, scl_time_corr_p = spearmanr(t, scl)
    else:
        raise ValueError("corr_method must be 'pearson' or 'spearman'")

    # 3) Get SCR event indices
    # NeuroKit2 stores indices in info, but keys may vary slightly depending on version.
    onsets = np.array(info.get("SCR_Onsets", []), dtype=float)
    peaks = np.array(info.get("SCR_Peaks", []), dtype=float)
    recovery = np.array(info.get("SCR_Recovery", []), dtype=float)

    # Remove NaNs / invalid values and cast to int
    onsets = onsets[np.isfinite(onsets)].astype(int)
    peaks = peaks[np.isfinite(peaks)].astype(int)
    recovery = recovery[np.isfinite(recovery)].astype(int)

    # Align events robustly (onset -> peak -> recovery)
    # We'll create matched segments where onset < peak < recovery if possible.
    segments = []
    used_recovery = set()

    for peak in peaks:
        onset_candidates = onsets[onsets < peak]
        if len(onset_candidates) == 0:
            continue
        onset = onset_candidates[-1]  # nearest onset before peak

        rec_candidates = recovery[(recovery > peak)]
        rec_candidates = [r for r in rec_candidates if r not in used_recovery]
        if len(rec_candidates) == 0:
            # If no recovery marker, estimate end at next time SCR returns near local baseline
            # Simple fallback: use peak + 4s (capped to signal length)
            rec = min(len(scr) - 1, int(peak + 4 * fs))
        else:
            rec = rec_candidates[0]
            used_recovery.add(rec)

        if onset < peak < rec:
            segments.append((onset, peak, rec))

    # 4) Number of SCR segments
    n_scr_segments = len(segments)

    # 5) SCR startle magnitudes and response durations
    # Startle magnitude here = peak amplitude relative to onset baseline (phasic component)
    # Duration = recovery - onset (seconds)
    startle_magnitudes = []
    response_durations = []
    scr_auc_list = []

    for onset, peak, rec in segments:
        magnitude = scr[peak] - scr[onset]
        duration = (rec - onset) / fs

        # Area under identified SCR segment (above onset baseline)
        segment_y = scr[onset:rec + 1] - scr[onset]
        # Optional: clip negative values so only positive response contributes
        segment_y = np.clip(segment_y, 0, None)
        segment_t = t[onset:rec + 1]
        auc = trapezoid(segment_y, segment_t)

        startle_magnitudes.append(magnitude)
        response_durations.append(duration)
        scr_auc_list.append(auc)

    startle_magnitudes = np.array(startle_magnitudes, dtype=float)
    response_durations = np.array(response_durations, dtype=float)
    scr_auc_list = np.array(scr_auc_list, dtype=float)

    # Sums requested by you
    sum_scr_startle_magnitudes = float(np.nansum(startle_magnitudes)) if len(startle_magnitudes) else 0.0
    sum_response_durations = float(np.nansum(response_durations)) if len(response_durations) else 0.0
    area_under_identified_scr = float(np.nansum(scr_auc_list)) if len(scr_auc_list) else 0.0

    # Additional useful summaries (optional)
    features = {
        "EDA_mean": np.mean(eda),
        "EDA_std": np.std(eda),
        "EDA_max": max(eda),
        "EDA_min": min(eda),
        "EDA_slope": np.polyfit(np.arange(len(eda)), eda, 1)[0],
        "EDA_range": max(eda) - min(eda),

        # Whole-signal tonic/phasic summaries
        "SCL_mean": float(np.nanmean(scl)),
        "SCL_std": float(np.nanstd(scl)),
        "SCR_mean": float(np.nanmean(scr)),
        "SCR_std": float(np.nanstd(scr)),

        # Correlation
        "SCL_time_corr": float(scl_time_corr),
        "SCL_time_corr_pvalue": float(scl_time_corr_p),

        # Event-based SCR features
        "num_SCR_segments": int(n_scr_segments),
        "sum_SCR_startle_magnitudes": sum_scr_startle_magnitudes,
        "sum_response_durations_sec": sum_response_durations,
        "area_under_identified_SCR": area_under_identified_scr,

        # Optional per-event stats
        "mean_SCR_startle_magnitude": float(np.nanmean(startle_magnitudes)) if len(startle_magnitudes) else 0.0,
        "mean_response_duration_sec": float(np.nanmean(response_durations)) if len(response_durations) else 0.0,
        "mean_SCR_auc": float(np.nanmean(scr_auc_list)) if len(scr_auc_list) else 0.0,
    }

    return features, signals, info

In [16]:
# =========================== Data processing: Time Series data with HRV ===================================================
x = []
y = []

for sid in SUBJECT_IDS:
    print(sid)
    subject = load_subject(f"{DATA_PATH}/{sid}/{sid}.pkl")
    #print(min((subject['signal']['chest']['Resp']/65536 - 0.5)*100))

    data_subject_indices.append(len(x))
    print(data_subject_indices)

    labels = np.array(subject['label'])
    labels = downsample_label(labels, 700, 1.0/LABEL_WINDOW)

    temp = np.array(subject['signal']['wrist']['TEMP'])

    bvp = np.array(subject['signal']['wrist']['BVP'])
    bvp_clean = nk.ppg_clean(bvp, sampling_rate=64)
    peaks, hrv_info = nk.ppg_peaks(bvp_clean, sampling_rate=64)

    acc = np.array(subject['signal']['wrist']['ACC'])

    eda = np.array(subject['signal']['wrist']['EDA'])

    for start in range(0, len(labels)):
        temp_s = temp[start * LABEL_WINDOW * 4: start * LABEL_WINDOW * 4 + INPUT_WINDOW * 4]
        bvp_s = bvp[start * LABEL_WINDOW * 64: start * LABEL_WINDOW * 64 + INPUT_WINDOW * 64]
        acc_s = acc[start * LABEL_WINDOW * 32: start * LABEL_WINDOW * 32 + INPUT_WINDOW * 32]
        label_s = labels[start]

        if np.count_nonzero(peaks[start * LABEL_WINDOW * 64: start * LABEL_WINDOW * 64 + INPUT_WINDOW * 64]) <= 5 or len(bvp_s) != INPUT_WINDOW * 64:
            continue

        hrv = []
        if USE_FREQ_HRV:
            hrv = nk.hrv(peaks[start * LABEL_WINDOW * 64: start * LABEL_WINDOW * 64 + INPUT_WINDOW * 64], sampling_rate=64, show=False)
        else:
            hrv = nk.hrv_time(peaks[start * LABEL_WINDOW * 64: start * LABEL_WINDOW * 64 + INPUT_WINDOW * 64], sampling_rate=64, show=False)
        

        data = [temp_s[:,0], bvp_s[:,0], acc_s[:,0], acc_s[:,1], acc_s[:,2], np.array(hrv["HRV_RMSSD"]), np.array(hrv["HRV_SDNN"]), np.array(hrv["HRV_pNN50"])]
        if USE_FREQ_HRV:
            data.append(np.array(hrv["HRV_LF"]))
            data.append(np.array(hrv["HRV_HF"]))
            data.append(np.array(hrv["HRV_LFHF"]))
        x_s = np.concatenate(data)

        shape_size = INPUT_WINDOW * 164 + 6 if USE_FREQ_HRV else INPUT_WINDOW * 164 + 3
        if x_s.shape[0] != shape_size:
            continue
        x.append(x_s)
        y.append(label_s)


x = np.array(x)
y = np.array(y)
print(x.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2, shuffle = True, stratify=y)
    

S2


/tmp/ipykernel_310169/167058565.py:16: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(f, encoding="latin1")


[-50.076]
(0,)
(0,)


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [27]:
# =========================== Data processing: WESAD Table ===================================================
x = []
y = []
data_subject_indices = []

for sid in SUBJECT_IDS:
    print(sid)
    data_subject_indices.append(len(x))
    print(data_subject_indices)
    subject = load_subject(f"{DATA_PATH}/{sid}/{sid}.pkl")
    survey = load_survey(f"{DATA_PATH}/{sid}/{sid}_readme.txt")

    labels = np.array(subject['label'])
    labels = downsample_label(labels, SAMPLE_FS['LABEL'], 1.0/LABEL_WINDOW)

    temp = np.array(subject['signal']['wrist']['TEMP'])

    bvp = np.array(subject['signal']['wrist']['BVP'])
    bvp_signals, bvp_info = nk.ppg_process(bvp, sampling_rate=64)

    acc = np.array(subject['signal']['wrist']['ACC'])
    #acc = filterSignalFIR(acc)

    eda = np.array(subject['signal']['wrist']['EDA']).squeeze(1)

    survey_vec = [int(survey[i].split(" ")[-1]) for i in range(1, 4)]

    if survey[4].split(" ")[-1] == "male\n":
        survey_vec += [1, 0]
    elif survey[4].split(" ")[-1] == "female\n":
        survey_vec += [0, 1]
    else:
        raise Exception("Unexpected gender")
    
    if survey[5].split(" ")[-1] == "right\n":
        survey_vec += [1, 0]
    elif survey[5].split(" ")[-1] == "left\n":
        survey_vec += [0, 1]
    else:
        raise Exception("Unexpected dominant hand")
    
    for i in range(8, 14):
        response = survey[i].split(" ")[-1]
        if response == "YES\n":
            survey_vec += [1, 0]
        elif response == "NO\n":
            survey_vec += [0, 1]
        else:
            raise Exception("Unexpected yes or no")
    print(survey_vec)

    for start in range(0, len(labels)):
        temp_s = temp[start * LABEL_WINDOW * SAMPLE_FS['TEMP']: start * LABEL_WINDOW * SAMPLE_FS['TEMP'] + INPUT_WINDOW * SAMPLE_FS['TEMP']]
        bvp_s = bvp[start * LABEL_WINDOW * SAMPLE_FS['BVP']: start * LABEL_WINDOW * SAMPLE_FS['BVP'] + INPUT_WINDOW * SAMPLE_FS['BVP']]
        peaks_s = np.array(bvp_signals['PPG_Peaks'][start * LABEL_WINDOW * SAMPLE_FS['BVP']: start * LABEL_WINDOW * SAMPLE_FS['BVP'] + INPUT_WINDOW * SAMPLE_FS['BVP']])
        if np.count_nonzero(peaks_s) <= 3 or len(bvp_s) != INPUT_WINDOW * SAMPLE_FS['BVP']:
            continue
        acc_s = acc[start * LABEL_WINDOW * SAMPLE_FS['ACC']: start * LABEL_WINDOW * SAMPLE_FS['ACC'] + INPUT_WINDOW * SAMPLE_FS['ACC']]
        acc_s = acc_s.reshape(160, 12, 3)
        eda_s = eda[start * LABEL_WINDOW * SAMPLE_FS['EDA']: start * LABEL_WINDOW * SAMPLE_FS['EDA'] + INPUT_WINDOW * SAMPLE_FS['EDA']]
        label_s = labels[start]

        acc_mean = np.mean(acc_s, axis = 0)
        acc_std = np.std(acc_s, axis = 0)
        acc_sum = np.trapezoid(np.abs(acc_s), axis = 0)
        acc_peak_freq = dom_nonzero_freq(acc_s, SAMPLE_FS['ACC'])
        acc_data = np.concatenate([acc_mean.flatten(), acc_std.flatten(), acc_sum.flatten(), acc_peak_freq.flatten()])

        temp_mean = np.mean(temp_s)
        temp_std = np.std(temp_s)
        temp_min = np.min(temp_s)
        temp_max = np.max(temp_s)
        temp_range = temp_max - temp_min
        temp_slope = np.polyfit(np.arange(len(temp_s)), temp_s, 1)[0][0]
        temp_data = [temp_mean, temp_std, temp_min, temp_max, temp_range, temp_slope]
        
        hrv = nk.hrv_time(peaks_s, sampling_rate=SAMPLE_FS['BVP'], show=False) if not USE_FREQ_HRV else nk.hrv(peaks_s, sampling_rate=SAMPLE_FS['BVP'], show=False)
        hr_mean = hrv['HRV_MeanNN']
        hr_std = hrv['HRV_SDNN']
        hrv_rmssd = hrv['HRV_RMSSD']
        hrv_pNN50 = hrv['HRV_pNN50']
        hrv_tinn = hrv['HRV_TINN']
        hrv_data = [hr_mean, hr_std, hrv_rmssd, hrv_pNN50, hrv_tinn]

        eda_feature, _, _ = eda_features(eda_s, SAMPLE_FS['EDA'])
        eda_data = [eda_feature["EDA_mean"], eda_feature["EDA_std"], eda_feature["EDA_max"], eda_feature["EDA_min"], eda_feature["EDA_slope"], eda_feature["EDA_range"], 
                    eda_feature["SCL_mean"], eda_feature["SCL_std"], eda_feature["SCR_mean"], eda_feature["SCR_std"], eda_feature["SCL_time_corr"], eda_feature["num_SCR_segments"],
                    eda_feature["sum_SCR_startle_magnitudes"], eda_feature["sum_response_durations_sec"], eda_feature["area_under_identified_SCR"]]

        if USE_FREQ_HRV:
            hrv_data.append(hrv['HRV_ULF'])
            hrv_data.append(hrv['HRV_LF'])
            hrv_data.append(hrv['HRV_HF'])
            hrv_data.append(hrv['HRV_VHF'])
            hrv_data.append(hrv['HRV_LFHF'])
            hrv_data.append(hrv['HRV_LFn'])
            hrv_data.append(hrv['HRV_HFn'])

        x_s = np.array(np.concatenate([acc_data, temp_data, np.array(hrv_data).squeeze(1), eda_data, np.array(survey_vec)]))

        x.append(x_s)
        y.append(label_s)
data_subject_indices.append(len(x))
print(data_subject_indices)

x = np.array(x)
y = np.array(y)
print(x.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2, shuffle = True, stratify=y)
    


S2
[0]


/tmp/ipykernel_310169/1584565278.py:14: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(f, encoding="latin1")


[27, 175, 80, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
S3
[0, 3010]


/tmp/ipykernel_310169/1584565278.py:14: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(f, encoding="latin1")


[27, 173, 69, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
S4
[0, 3010, 6183]


...

[28, 186, 83, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
S16
[0, 3010, 6183, 9365, 12376, 15882, 18472, 21176, 23758, 26477, 29064, 31803, 34548, 37075]


/tmp/ipykernel_310169/1584565278.py:14: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(f, encoding="latin1")


[24, 184, 69, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
S17
[0, 3010, 6183, 9365, 12376, 15882, 18472, 21176, 23758, 26477, 29064, 31803, 34548, 37075, 39861]


/tmp/ipykernel_310169/1584565278.py:14: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(f, encoding="latin1")


[29, 165, 55, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
[0, 3010, 6183, 9365, 12376, 15882, 18472, 21176, 23758, 26477, 29064, 31803, 34548, 37075, 39861, 42792]
(42792, 189)
(42792,)


In [28]:
np.save("x_extracted_no_freq_5sACC_withsurvey.npy", x)
np.save("y_extracted_no_freq_5sACC_withsurvey.npy", y)

In [ ]:
# ================ Load preprocessed data ==========================================
x = np.load("../data/Processed/x_extracted_no_freq.npy")
y = np.load("../data/Processed/y_extracted_no_freq.npy")
print(x.shape)
print(y.shape)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2, shuffle = True, stratify=y)
# X_train = x[:34233]
# y_train = y[:34233]
# X_test = x[34233:]
# y_test = y[34233:]


In [ ]:
# ===================TEST XGB ======================================
xgb = XGBClassifier(
    n_estimators=30,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
    tree_method = 'hist',
    eval_metric=["mlogloss", "merror"])

xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
)

unique, counts = np.unique(y.copy(), return_counts=True)
pred = xgb.predict(X_test)
cm = confusion_matrix(y_test, pred)
f1 = f1_score(np.array(y_test), np.array(pred), average='micro', labels=unique)
print(f1)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=[f"Predicted {i}" for i in range(len(unique))],
    yticklabels=[f"Actual {i}" for i in range(len(unique))]
)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(x.shape[1], 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16), 
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 3),
        )

    def forward(self, x):
        return self.net(x)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2, padding=0)
        self.conv2 = nn.Conv1d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()

        self.classifier = nn.Sequential(
            nn.Flatten(),                  
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(128, 3)
        )


    def forward(self, x):
        x = self.relu(self.conv1(x.unsqueeze(dim=1)))
        print(x.shape)
        x = self.pool(x)
        print(x.shape)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        print(x.shape)

        return self.classifier(x)

In [ ]:
# ========================= TEST NN ==============================
device = "cuda:0"

def per_class_precision_recall_from_cm(cm):
    cm = np.asarray(cm)
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) != 0)
    recall    = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) != 0)
    return precision, recall

def get_data_loaders(subject_id):
    print(subject_id)
    train_start = x[:data_subject_indices[subject_id],:] if x[:data_subject_indices[subject_id],:].shape[0] != 0 else np.empty((0,x.shape[1]))
    train_end = x[data_subject_indices[subject_id + 1]:,:] if x[data_subject_indices[subject_id + 1]:,:].shape[0] != 0 else np.empty((0,x.shape[1]))
    train_subject = np.concatenate([train_start.copy(), train_end.copy()])
    
    test_subject = x[data_subject_indices[subject_id]:data_subject_indices[subject_id+1],:].copy()

    train_label_start = y[:data_subject_indices[subject_id]] if data_subject_indices[subject_id] > 0 else np.empty(0)
    train_label_end = y[data_subject_indices[subject_id + 1]:] if y[data_subject_indices[subject_id + 1]:].shape[0] != 0 else np.empty((0))
    train_label = np.concatenate([train_label_start.copy(), train_label_end.copy()])

    test_label = y[data_subject_indices[subject_id]:data_subject_indices[subject_id+1]].copy()
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_subject)
    X_test_scaled = scaler.transform(test_subject)
    train_dataset = TensorDataset(torch.tensor(X_train_scaled), torch.tensor(train_label,dtype=int))
    test_dataset = TensorDataset(torch.tensor(X_test_scaled), torch.tensor(test_label,dtype=int))

    return train_dataset, test_dataset

def eval(data, nn_model, log = False):
    nn_model.eval()
    correct = 0
    total = 0

    y_test = []
    pred = []

    with torch.no_grad():
        for tx, ty in data:
            tx, ty = tx.to(device), ty.to(device)

            logits = nn_model(tx)
            preds = logits.argmax(dim=1)

            y_test.extend(ty.cpu())
            pred.extend(preds.cpu())

            correct += (preds == ty).sum().item()
            total += ty.size(0)

    accuracy = correct / total
    
    if log:
        print(f"Test accuracy: {accuracy:.3f}")
        cm = confusion_matrix(np.array(y_test), np.array(pred), labels=unique)
        f1 = f1_score(np.array(y_test), np.array(pred), average='micro', labels=unique)
        print(f"F1: {f1:.3f}")
        print(cm)

        precision, recall = per_class_precision_recall_from_cm(cm)
        print(f"Precision: {precision}   Recall:{recall}")
        # sns.heatmap(
        #     cm, annot=True, fmt="d", cmap="Blues",
        #     xticklabels=[f"Predicted {i}" for i in range(len(unique))],
        #     yticklabels=[f"Actual {i}" for i in range(len(unique))]
        # )

for i in range(len(SUBJECT_IDS)):
    model = MLP().to(torch.float64).to(device)

    tr_dataset, te_dataset = get_data_loaders(i)
    dataloader_tr = DataLoader(tr_dataset, batch_size=256, shuffle=True)
    dataloader_te = DataLoader(te_dataset, batch_size=256)

    te_arr = te_dataset.tensors[1].detach().cpu().numpy()

    unique, counts = np.unique(te_arr.copy(), return_counts=True)
    class_weight = torch.tensor([len(te_arr) / (len(unique) * counts[i]) for i in range(len(unique))]).to(torch.float64).to(device)
    print(class_weight)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    num_epochs = 1000

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        for batch_x, batch_y in dataloader_tr:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            # Forward pass
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            if torch.isnan(batch_x).any() or torch.isinf(batch_x).any():
                print('invalid input detected')

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if epoch % 100 == 0:
            eval(dataloader_te, model, True)
            avg_loss = total_loss / len(dataloader_tr)
            print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.4f}")
    eval(dataloader_te, model, True)



In [3]:
data = pd.read_parquet("../data/Dapper/dapper_dqn_10s_step_62min_norm.parquet")

In [ ]:
data['High Valence'] = (data['Valence'] > data.groupby('PID')['Valence'].transform('mean')).astype(int)
data['High Arousal'] = (data['Arousal'] > data.groupby('PID')['Arousal'].transform('mean')).astype(int)

print(data["High Arousal"])

0         0
1         0
2         0
3         0
4         0
         ..
644248    1
644249    1
644250    1
644251    1
644252    1
Name: High Arousal, Length: 644253, dtype: int64


In [20]:
data.to_parquet('dapper_dqn_10s_step_62min_norm_w_subject_mean_arousal_valence.parquet')

In [21]:
print(len(data))

644253
